<a href="https://colab.research.google.com/github/Sathvik071404/BMSCE_BIS/blob/main/Drone_Route_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

def calculate_cost(paths, start, end, obstacles, obstacle_radius, w_length=1.0, w_obs=1000.0):
    """
    Highly vectorized cost calculation.
    paths shape: (n_particles, n_waypoints, 3)
    """
    n_particles, n_waypoints, _ = paths.shape

    # 1. Path Length Cost (Distance between sequential waypoints including start and end)
    full_paths = np.concatenate([np.tile(start, (n_particles, 1, 1)), paths, np.tile(end, (n_particles, 1, 1))], axis=1)
    segments = np.diff(full_paths, axis=1)
    lengths = np.sum(np.sqrt(np.sum(segments**2, axis=2)), axis=1)

    # 2. Obstacle Penalty (Vectorized distance from every waypoint to every obstacle)
    # paths[:, :, None, :] shape: (n_particles, n_waypoints, 1, 3)
    # obstacles[None, None, :, :] shape: (1, 1, n_obstacles, 3)
    dist_to_obs = np.sqrt(np.sum((paths[:, :, None, :] - obstacles[None, None, :, :])**2, axis=3))

    # Check if distance is less than safety radius
    collisions = dist_to_obs < obstacle_radius[None, None, :]
    obs_penalty = np.sum(collisions, axis=(1, 2)) * w_obs

    return w_length * lengths + obs_penalty

def optimize_drone_route(start, end, obstacles, obstacle_radius, n_particles=40, n_waypoints=5, max_iter=100):
    # Hyperparameters
    w, w_decay = 0.9, 0.98
    c1, c2 = 1.5, 1.5
    v_max = 2.0

    # Initialize positions linearly between start and end with random noise
    t = np.linspace(0, 1, n_waypoints + 2)[1:-1]
    line_path = start + t[:, None] * (end - start)

    pos = np.tile(line_path, (n_particles, 1, 1))
    pos += np.random.uniform(-5, 5, size=pos.shape)

    vel = np.random.uniform(-v_max, v_max, size=pos.shape)

    # Track historical bests
    pbest_pos = pos.copy()
    pbest_cost = calculate_cost(pos, start, end, obstacles, obstacle_radius)

    gbest_idx = np.argmin(pbest_cost)
    gbest_pos = pbest_pos[gbest_idx].copy()
    gbest_cost = pbest_cost[gbest_idx]

    # Main Optimization Loop
    for _ in range(max_iter):
        w *= w_decay
        r1 = np.random.rand(n_particles, 1, 1)
        r2 = np.random.rand(n_particles, 1, 1)

        # Vectorized swarm velocity & position update
        vel = (w * vel) + (c1 * r1 * (pbest_pos - pos)) + (c2 * r2 * (gbest_pos[None, :, :] - pos))
        vel = np.clip(vel, -v_max, v_max)
        pos += vel

        # Cost evaluation
        current_cost = calculate_cost(pos, start, end, obstacles, obstacle_radius)

        # Update Personal Best
        better_mask = current_cost < pbest_cost
        pbest_cost[better_mask] = current_cost[better_mask]
        pbest_pos[better_mask] = pos[better_mask]

        # Update Global Best
        min_idx = np.argmin(pbest_cost)
        if pbest_cost[min_idx] < gbest_cost:
            gbest_cost = pbest_cost[min_idx]
            gbest_pos = pbest_pos[min_idx].copy()

    # Return complete route: Start -> Optimized Waypoints -> End
    return np.vstack([start, gbest_pos, end])

# --- Execution Example ---
if __name__ == "__main__":
    start_point = np.array([0.0, 0.0, 0.0])
    end_point = np.array([50.0, 50.0, 20.0])

    # 3D Spherical obstacles: [X, Y, Z]
    obs_centers = np.array([
        [15.0, 15.0, 5.0],
        [30.0, 32.0, 12.0]
    ])
    obs_radii = np.array([6.0, 8.0]) # Clearance size for each obstacle

    best_route = optimize_drone_route(start_point, end_point, obs_centers, obs_radii)
    print("Optimized Waypoints:\n", best_route)

Optimized Waypoints:
 [[ 0.          0.          0.        ]
 [ 7.08695207  7.86024087  2.12371858]
 [18.41139704 19.64248074  6.82641484]
 [24.44082956 25.65345309  8.76033922]
 [37.00483193 37.64684017 14.42325995]
 [41.39337748 41.85852135 16.12741089]
 [50.         50.         20.        ]]


In [ ]:
Set max_iterations, swarm_size, w, c1, c2
Define start_position, target_position, obstacles

For i = 1 To swarm_size:
    particle[i].position = Generate_Path(start_position, target_position)
    particle[i].velocity = Init_Velocity()
    particle[i].cost = Cost(particle[i].position, obstacles)
    particle[i].pbest_pos = particle[i].position
    particle[i].pbest_cost = particle[i].cost

gbest_idx = argmin(particle.pbest_cost)
gbest_pos = particle[gbest_idx].pbest_pos
gbest_cost = particle[gbest_idx].pbest_cost

For iter = 1 To max_iterations:
    w = w * w_decay // Dynamic inertia reduction

    For i = 1 To swarm_size:
        r1, r2 = Random(0, 1), Random(0, 1)

        // Vectorised waypoint update
        particle[i].velocity = (w * particle[i].velocity)
                             + (c1 * r1 * (particle[i].pbest_pos - particle[i].position))
                             + (c2 * r2 * (gbest_pos - particle[i].position))

        Clamp(particle[i].velocity, -v_max, v_max)
        particle[i].position += particle[i].velocity

        // Analytical vector cost calculation
        current_cost = Fast_Cost(particle[i].position, obstacles)

        If current_cost < particle[i].pbest_cost Then:
            particle[i].pbest_cost = current_cost
            particle[i].pbest_pos = particle[i].position

            If current_cost < gbest_cost Then:
                gbest_cost = current_cost
                gbest_pos = particle[i].position

Return gbest_pos
